# 04 · Evaluate

1. Kaggle: **Add-ons → Secrets** → tick `HF_TOKEN` for this notebook.
2. Settings → Accelerator → **GPU T4 x2**, Internet **On**.
3. **Save Version → Save & Run All (Commit)**. Do not use Run All.

Results for each model are uploaded to `<your-username>/legal-llama-training/evaluation/` as soon as that model finishes. If the run stops, start it again the same way: finished models are skipped.

In [ ]:
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel
import os, re, gc, json, time, random, string
from bisect import bisect_right
from collections import Counter
import torch, pandas as pd
from huggingface_hub import HfApi, hf_hub_download, list_repo_files, snapshot_download
assert torch.cuda.is_available(), "No GPU. Kaggle: Settings > Accelerator > GPU T4 x2 (and Internet: On). Colab: Runtime > Change runtime type > T4 GPU"

def get_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

HF_TOKEN = get_token()
assert HF_TOKEN, "HF_TOKEN secret missing. Kaggle: Add-ons > Secrets > add HF_TOKEN and tick it for this notebook."
api = HfApi(token=HF_TOKEN)
SAVE_REPO = f"{api.whoami()['name']}/legal-llama-training"
assert "final/adapter_model.safetensors" in api.list_repo_files(SAVE_REPO), f"No trained adapter found in {SAVE_REPO}/final"
ADAPTER_DIR = os.path.join(snapshot_download(SAVE_REPO, allow_patterns=["final/*"], token=HF_TOKEN), "final")
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
SEED = 3407
print("Trained adapter found:", ADAPTER_DIR)

In [ ]:
DATA_REPO = "theatticusproject/cuad-qa"
DATA_REV = "refs/convert/parquet"
paths = [
    hf_hub_download(DATA_REPO, f, repo_type="dataset", revision=DATA_REV)
    for f in list_repo_files(DATA_REPO, repo_type="dataset", revision=DATA_REV)
    if f.startswith("default/test/") and f.endswith(".parquet")
]
test = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
test["answer_texts"] = test["answers"].apply(lambda a: list(a["text"]))
test["answer_starts"] = test["answers"].apply(lambda a: [int(s) for s in a["answer_start"]])
test["has_answer"] = test["answer_texts"].apply(len) > 0
test = test.drop(columns=["answers"])
print(len(test), "test rows from", test.title.nunique(), "contracts")

In [ ]:
BASE = "unsloth/llama-3-8b-bnb-4bit"
OLD = "CSroseX/Legal_llama3.1_part5"
MAX_TOKENS = 2048
WINDOW = 512
MAX_WINDOW = 1024
N_POSITIVE = 500
N_NEGATIVE = 500
NOT_FOUND = "Not found."
ALPACA = (
    "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{}\n\n### Input:\n{}\n\n### Response:\n"
)
INSTRUCTION = (
    "{question}\n"
    "Quote the exact text from the contract excerpt that answers this, one passage per line. "
    'If there is none, answer "Not found."'
)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE)

contracts = test.drop_duplicates("title").set_index("title")["context"].to_dict()
token_starts, token_ends = {}, {}
for t, c in contracts.items():
    offsets = tokenizer(c, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
    token_starts[t] = [s for s, _ in offsets]
    token_ends[t] = [e for _, e in offsets]

def token_index(t, char):
    return max(0, bisect_right(token_starts[t], char) - 1)

def answer_spans(row):
    return [(s, s + len(a), a) for a, s in zip(row.answer_texts, row.answer_starts)]

def make(row, start, end):
    a, b = token_starts[row.title][start], token_ends[row.title][end - 1]
    inside = []
    for s, e, text in answer_spans(row):
        if a <= s and e <= b:
            inside.append((s, text))
        elif s < b and a < e:
            return None
    answers = list(dict.fromkeys(text for _, text in sorted(inside)))
    return {"title": row.title, "question": row.question, "input": contracts[row.title][a:b], "output": "\n".join(answers) if answers else NOT_FOUND}

def positive(row, rng):
    t = row.title
    n = len(token_starts[t])
    s, e, _ = answer_spans(row)[0]
    i0, i1 = token_index(t, s), token_index(t, e - 1) + 1
    size = max(WINDOW, i1 - i0 + 64)
    if size > MAX_WINDOW:
        return None
    size = min(size, n)
    lo, hi = max(0, i1 - size), min(i0, n - size)
    start = rng.randint(lo, hi) if lo <= hi else max(0, n - size)
    return make(row, start, start + size)

def negative(row, rng, tries=20):
    n = len(token_starts[row.title])
    size = min(WINDOW, n)
    for _ in range(tries):
        start = rng.randint(0, n - size)
        ex = make(row, start, start + size)
        if ex and ex["output"] == NOT_FOUND:
            return ex
    return None

rng = random.Random(SEED)
answered, unanswered = test[test.has_answer], test[~test.has_answer]
pos = [x for x in (positive(r, rng) for r in answered.sample(frac=1, random_state=SEED).itertuples()) if x][:N_POSITIVE]
neg_pool = (
    list(unanswered.sample(min(N_NEGATIVE, len(unanswered)), random_state=SEED).itertuples())
    + list(answered.sample(min(N_NEGATIVE, len(answered)), random_state=SEED).itertuples())
)
rng.shuffle(neg_pool)
neg = [x for x in (negative(r, rng) for r in neg_pool) if x][:N_NEGATIVE]
exam = pos + neg
print(len(pos), "pages with an answer,", len(neg), "pages without")

In [ ]:
def clause_of(question):
    m = re.search(r'related to "([^"]+)"', question)
    return m.group(1) if m else "?"

def norm(s):
    s = "".join(ch for ch in s.lower() if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def word_overlap(pred, gold):
    p, g = norm(pred).split(), norm(gold).split()
    if not p or not g:
        return float(p == g)
    same = sum((Counter(p) & Counter(g)).values())
    if same == 0:
        return 0.0
    precision, recall = same / len(p), same / len(g)
    return 2 * precision * recall / (precision + recall)

def is_not_found(pred):
    return norm(pred).startswith("not found")

def squash(s):
    return " ".join(s.lower().split())

def quoted(pred, page):
    lines = [l.strip().strip('"') for l in pred.split("\n") if l.strip()]
    return bool(lines) and all(squash(l) in squash(page) for l in lines)

def score(examples, preds):
    rows = []
    for ex, p in zip(examples, preds):
        gold_nf, pred_nf = ex["output"] == NOT_FOUND, is_not_found(p)
        rows.append({
            "clause": clause_of(ex["question"]),
            "gold": ex["output"],
            "pred": p,
            "gold_not_found": gold_nf,
            "pred_not_found": pred_nf,
            "overlap": None if gold_nf else (0.0 if pred_nf else word_overlap(p, " ".join(ex["output"].split("\n")))),
            "quoted": None if pred_nf else quoted(p, ex["input"]),
        })
    df = pd.DataFrame(rows)
    pos, neg, said = df[~df.gold_not_found], df[df.gold_not_found], df[~df.pred_not_found]
    summary = {
        "word_overlap": round(pos.overlap.mean(), 3),
        "answers_found": round((pos.overlap >= 0.5).mean(), 3),
        "missed": round(pos.pred_not_found.mean(), 3),
        "correct_not_found": round(neg.pred_not_found.mean(), 3),
        "not_quoted_from_contract": round(1 - said.quoted.astype(bool).mean(), 3) if len(said) else 0.0,
    }
    by_clause = pos.groupby("clause").overlap.agg(["mean", "count"]).round(3).sort_values("mean", ascending=False)
    return summary, by_clause, rows

def generate(model, tok, prompts, max_new_tokens=200, batch_size=8):
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    results = [None] * len(prompts)
    for b in range(0, len(order), batch_size):
        idx = order[b : b + batch_size]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                eos_token_id=tok.eos_token_id,
                pad_token_id=tok.pad_token_id,
            )
        for j, i in enumerate(idx):
            results[i] = tok.decode(out[j, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return results

def new_prompt(question, page):
    return ALPACA.format(INSTRUCTION.format(question=question), page)

def old_prompt(question, page):
    return ALPACA.format(question, page)

def run(name, model_id, prompt_fn, extra=None):
    path = f"evaluation/{name}.json"
    if path in api.list_repo_files(SAVE_REPO):
        print(name, "already evaluated, loading saved result")
        return json.load(open(hf_hub_download(SAVE_REPO, path, token=HF_TOKEN)))
    start = time.time()
    model, tok = FastLanguageModel.from_pretrained(model_name=model_id, max_seq_length=MAX_TOKENS, load_in_4bit=True, dtype=None, token=HF_TOKEN)
    FastLanguageModel.for_inference(model)
    preds = generate(model, tok, [prompt_fn(x["question"], x["input"]) for x in exam])
    summary, by_clause, rows = score(exam, preds)
    result = {"model": model_id, "summary": summary, "by_clause": by_clause.reset_index().to_dict("records"), "rows": rows}
    if extra:
        result["full_contract"] = extra(model, tok)
    result["minutes"] = round((time.time() - start) / 60, 1)
    del model, tok
    gc.collect()
    torch.cuda.empty_cache()
    local = f"{OUT}/{name}.json"
    json.dump(result, open(local, "w"), ensure_ascii=False)
    for attempt in range(3):
        try:
            api.upload_file(path_or_fileobj=local, path_in_repo=path, repo_id=SAVE_REPO, commit_message=f"evaluation {name}")
            break
        except Exception as e:
            print("upload failed, retrying:", e)
            time.sleep(30)
    print(name, json.dumps(summary), f"{result['minutes']} min")
    return result

In [ ]:
OVERLAP = 64
DEMO_CONTRACTS = 3
DEMO_CLAUSES = ["Parties", "Agreement Date", "Governing Law", "Expiration Date", "Anti-Assignment", "Non-Compete"]

def pages_of(text, tok, window=WINDOW, overlap=OVERLAP):
    offsets = tok(text, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
    n = len(offsets)
    starts = [s for s in range(0, n, window - overlap) if s == 0 or s < n - overlap]
    return [text[offsets[s][0] : offsets[min(s + window, n) - 1][1]] for s in starts]

def find_in_contract(model, tok, text, question):
    pages = pages_of(text, tok)
    outs = generate(model, tok, [new_prompt(question, p) for p in pages])
    found = []
    for o in outs:
        if is_not_found(o):
            continue
        for line in o.split("\n"):
            line = line.strip()
            if line and line not in found:
                found.append(line)
    return found

def full_contract_demo(model, tok):
    titles = sorted(contracts, key=lambda t: len(token_starts[t]))
    titles = titles[len(titles) // 4 : len(titles) // 4 + DEMO_CONTRACTS]
    results = []
    for t in titles:
        for clause in DEMO_CLAUSES:
            row = test[(test.title == t) & (test.question.map(clause_of) == clause)]
            if row.empty:
                continue
            row = row.iloc[0]
            found = find_in_contract(model, tok, contracts[t], row.question)
            gold = row.answer_texts
            if gold:
                ok = round(word_overlap(" ".join(found), " ".join(gold)), 3) if found else 0.0
            else:
                ok = 1.0 if not found else 0.0
            results.append({"contract": t, "tokens": len(token_starts[t]), "clause": clause, "expected": gold or [NOT_FOUND], "found": found or [NOT_FOUND], "score": ok})
            print(json.dumps(results[-1], ensure_ascii=False)[:400])
    return results

In [ ]:
results = {}
results["base"] = run("base", BASE, new_prompt)

In [ ]:
results["new"] = run("new", ADAPTER_DIR, new_prompt, extra=full_contract_demo)

In [ ]:
results["old"] = run("old", OLD, old_prompt)

In [ ]:
table = pd.DataFrame({name: r["summary"] for name, r in results.items()})[["base", "old", "new"]]
print(table.to_string())
print()
by_clause = pd.DataFrame(results["new"]["by_clause"])
print("New model, best clause types:")
print(by_clause.head(8).to_string(index=False))
print("New model, worst clause types:")
print(by_clause.tail(8).to_string(index=False))
print()
demo = pd.DataFrame(results["new"]["full_contract"])
print("Full-contract test, average score:", round(demo.score.mean(), 3))
print(demo[["contract", "tokens", "clause", "score"]].to_string(index=False))
report = {
    "exam": {"pages_with_answer": len(pos), "pages_without": len(neg), "contracts": test.title.nunique()},
    "summary": table.to_dict(),
    "best_clauses": by_clause.head(8).to_dict("records"),
    "worst_clauses": by_clause.tail(8).to_dict("records"),
    "full_contract": results["new"]["full_contract"],
}
json.dump(report, open(f"{OUT}/evaluation_report.json", "w"), ensure_ascii=False, indent=1)
api.upload_file(path_or_fileobj=f"{OUT}/evaluation_report.json", path_in_repo="evaluation/report.json", repo_id=SAVE_REPO, commit_message="evaluation report")
print(json.dumps(report, indent=1, ensure_ascii=False)[:6000])